# LLM Fine-Tuning: Customer Support Agent

Fine-tunes **Llama 3.2 3B Instruct** on 27K real customer support conversations using LoRA.

**Runtime**: T4 GPU (free Colab tier)  
**Time**: ~30 min for 200 steps  
**Dataset**: [Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset)

## Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Left sidebar 🔑 Secrets → add `WANDB_API_KEY` and `HF_TOKEN`
3. Run all cells top to bottom (Ctrl+F9)

## What this notebook does:
- **Cells 1–9**: Train the fine-tuned model
- **Cell 10**: Save model + push to HuggingFace Hub
- **Cell 11**: Quick sanity-check inference
- **Cell 12**: Generate predictions for BOTH base and fine-tuned models
- **Cell 13**: Download everything (predictions + test set + model zip)

After downloading, run locally:
```bash
python evaluation/compute_metrics.py --experiment r16
python evaluation/llm_judge.py --experiment r16
```

In [ ]:
# ── cell 1: check GPU ──────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_available(), 'Switch to T4 GPU: Runtime → Change runtime type'

In [ ]:
# ── cell 2: install dependencies ───────────────────────────────────────
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install wandb datasets evaluate rouge-score rich python-dotenv scikit-learn

In [ ]:
# ── cell 3: auth ───────────────────────────────────────────────────────
import os
from google.colab import userdata
import wandb

# Keys must be added in Colab Secrets (left sidebar 🔑) — NOT pasted here
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['HF_TOKEN']      = userdata.get('HF_TOKEN')

wandb.login()
print('Auth complete')

In [ ]:
# ── cell 4: config — only cell you need to change between experiments ──
MAX_SEQ_LENGTH = 2048
MODEL_NAME     = 'unsloth/Llama-3.2-3B-Instruct'
LORA_RANK      = 16    # ← change to 64 for second experiment
MAX_STEPS      = 200   # ← increase to 500+ for full training
N_EVAL_SAMPLES = 100   # predictions to generate after training

EXPERIMENT_NAME = f'customer-support-lora-r{LORA_RANK}'
print(f'Experiment : {EXPERIMENT_NAME}')
print(f'Max steps  : {MAX_STEPS}')
print(f'Eval samples: {N_EVAL_SAMPLES}')

In [ ]:
# ── cell 5: load base model with Unsloth ───────────────────────────────
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template='llama-3')
print(f'Loaded: {MODEL_NAME}')

In [ ]:
# ── cell 6: apply LoRA ─────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

In [ ]:
# ── cell 7: download + clean dataset ──────────────────────────────────
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

SYSTEM_PROMPT = (
    'You are a helpful, professional customer support agent. '
    'Respond clearly and empathetically to customer inquiries. '
    'Be concise, accurate, and solution-focused.'
)

ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
df = ds.to_pandas()
print(f'Raw: {len(df):,} rows')

df = df.dropna(subset=['instruction', 'response'])
df = df[df['response'].str.len() >= 20]
df = df[df['instruction'].str.len() >= 5]
df['instruction'] = df['instruction'].str.strip().str.replace(r'\s+', ' ', regex=True)
df['response']    = df['response'].str.strip().str.replace(r'\s+', ' ', regex=True)
df = df.drop_duplicates(subset=['instruction', 'response']).reset_index(drop=True)
print(f'After cleaning: {len(df):,} rows')

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['intent'], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, stratify=temp_df['intent'], random_state=42)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

def make_messages(row):
    return {'messages': [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': row['instruction']},
        {'role': 'assistant', 'content': row['response']},
    ]}

train_ds = Dataset.from_list([make_messages(r) for r in train_df.to_dict('records')])
val_ds   = Dataset.from_list([make_messages(r) for r in val_df.to_dict('records')])

def apply_template(examples):
    return {'text': tokenizer.apply_chat_template(
        examples['messages'], tokenize=False, add_generation_prompt=False
    )}

train_ds = train_ds.map(apply_template, batched=True)
val_ds   = val_ds.map(apply_template, batched=True)
print('Dataset ready.')
print(f'Sample:\n{train_ds[0]["text"][:300]}')

In [ ]:
# ── cell 8: configure trainer ──────────────────────────────────────────
import wandb
from trl import SFTTrainer
from transformers import TrainingArguments

wandb.init(
    project='llm-finetuning-customer-support',
    name=EXPERIMENT_NAME,
    config={'model': MODEL_NAME, 'lora_rank': LORA_RANK, 'max_steps': MAX_STEPS},
)

training_args = TrainingArguments(
    output_dir                  = f'outputs/{EXPERIMENT_NAME}',
    max_steps                   = MAX_STEPS,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps                = 10,
    learning_rate               = 2e-4 if LORA_RANK <= 16 else 1e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    eval_strategy               = 'steps',
    eval_steps                  = 50,
    save_steps                  = 100,
    seed                        = 42,
    optim                       = 'adamw_8bit',
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    report_to                   = 'wandb',
    run_name                    = EXPERIMENT_NAME,
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    eval_dataset       = val_ds,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args               = training_args,
)
print('Trainer ready.')

In [ ]:
# ── cell 9: TRAIN ──────────────────────────────────────────────────────
import time
start = time.time()
trainer.train()
elapsed = (time.time() - start) / 60
print(f'\nTraining done in {elapsed:.1f} minutes')
wandb.log({'train/total_minutes': elapsed})
wandb.finish()

In [ ]:
# ── cell 10: save model ────────────────────────────────────────────────
import os

model.save_pretrained(f'outputs/{EXPERIMENT_NAME}')
tokenizer.save_pretrained(f'outputs/{EXPERIMENT_NAME}')
print(f'Saved to outputs/{EXPERIMENT_NAME}')

# Push to HuggingFace Hub (requires HF_TOKEN secret with Write access)
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    repo_id = f'vamsiyvk/{EXPERIMENT_NAME}'
    try:
        model.push_to_hub(repo_id, token=hf_token)
        tokenizer.push_to_hub(repo_id, token=hf_token)
        print(f'Pushed to: https://huggingface.co/{repo_id}')
    except Exception as e:
        print(f'Hub push failed: {e}')
        print('Model is still saved locally — download the zip in cell 13.')
else:
    print('HF_TOKEN not set — skipping Hub push')

In [ ]:
# ── cell 11: quick sanity-check inference ──────────────────────────────
FastLanguageModel.for_inference(model)

def chat(instruction):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': instruction},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

for q in [
    'I was charged twice for my order.',
    'How do I return something I bought last week?',
    'My account is locked and I cannot log in.',
]:
    print(f'Q: {q}\nA: {chat(q)}\n{"─"*60}')

In [ ]:
# ── cell 12: generate predictions for BOTH models ─────────────────────
# Runs inference here on GPU so you never need to run the model locally.
# Saves two parquets → download them → compute_metrics.py does the rest.

import time
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTokenizer

def generate_predictions(gen_model, gen_tokenizer, dataframe, label, n=N_EVAL_SAMPLES):
    sample = dataframe.sample(n=n, random_state=42).reset_index(drop=True)
    predictions, latencies = [], []
    for _, row in sample.iterrows():
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': row['instruction']},
        ]
        try:
            prompt = gen_tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            prompt = f"{SYSTEM_PROMPT}\n\nUser: {row['instruction']}\nAssistant:"

        inputs = gen_tokenizer(prompt, return_tensors='pt').to('cuda')
        t0 = time.perf_counter()
        with torch.no_grad():
            out = gen_model.generate(
                **inputs, max_new_tokens=256, do_sample=False,
                pad_token_id=gen_tokenizer.eos_token_id,
            )
        latencies.append((time.perf_counter() - t0) * 1000)
        predictions.append(
            gen_tokenizer.decode(
                out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            ).strip()
        )

    result = sample[['instruction', 'response', 'intent', 'category']].copy()
    result['prediction'] = predictions
    result['latency_ms'] = latencies
    avg = sum(latencies) / len(latencies)
    print(f'{label}: {len(result)} samples | avg latency {avg:.0f} ms')
    return result

# — fine-tuned (already loaded in memory)
print('Generating fine-tuned predictions...')
ft_preds = generate_predictions(model, tokenizer, test_df, 'Fine-tuned')
ft_preds.to_parquet('finetuned_predictions.parquet', index=False)
print('Saved finetuned_predictions.parquet')

# — base model (load separately)
print('\nLoading base model for comparison...')
base_tok = HFTokenizer.from_pretrained(MODEL_NAME)
base_tok.pad_token = base_tok.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16
).to('cuda')
base_model.eval()

print('Generating base model predictions...')
base_preds = generate_predictions(base_model, base_tok, test_df, 'Base')
base_preds.to_parquet('base_predictions.parquet', index=False)
print('Saved base_predictions.parquet')

del base_model
torch.cuda.empty_cache()
print('\nBoth prediction files ready — run cell 13 to download everything.')

In [ ]:
# ── cell 13: download everything ───────────────────────────────────────
import shutil
from google.colab import files

# 1. predictions (needed for compute_metrics.py)
files.download('finetuned_predictions.parquet')
files.download('base_predictions.parquet')

# 2. test set
test_df.to_parquet('test_set.parquet', index=False)
files.download('test_set.parquet')

# 3. model zip (if Hub push failed)
print('Zipping model (takes ~1 min)...')
shutil.make_archive(EXPERIMENT_NAME, 'zip', f'outputs/{EXPERIMENT_NAME}')
files.download(f'{EXPERIMENT_NAME}.zip')

print(f"""
Place downloaded files at:
  finetuned_predictions.parquet  →  evaluation/results/r{LORA_RANK}/finetuned_predictions.parquet
  base_predictions.parquet       →  evaluation/results/r{LORA_RANK}/base_predictions.parquet
  test_set.parquet               →  data/cleaned/test_set.parquet
  {EXPERIMENT_NAME}.zip          →  unzip into outputs/{EXPERIMENT_NAME}/

Then locally:
  python evaluation/compute_metrics.py --experiment r{LORA_RANK}
  python evaluation/llm_judge.py --experiment r{LORA_RANK}
""")